In [3]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from scipy import stats
from xgboost import XGBRegressor
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    RandomizedSearchCV,
    cross_val_predict
)
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    explained_variance_score
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

SEED = 42
PATH = Path(r"D:\Raw data.csv")
PATH = PATH if PATH.exists() else Path(r"D:\Raw data")
OUT = Path(r"D:\BFO_XGBoost_Results")

for folder in ["Figures", "Tables", "Models"]:
    (OUT / folder).mkdir(parents=True, exist_ok=True)

TARGET = "Efficiency (%)"
CATEGORICAL = ["Catalyst_type", "Anions"]
NUMERIC = [
    "Surface area",
    "Pore Volume",
    "Catalyst_loading (g/L)",
    "Light_intensity (W)",
    "time (min)",
    "solution_pH",
    "HA (mg/L)",
    "Ci (mg/L)"
]
FEATURES = CATEGORICAL + NUMERIC
GROUP_COLUMNS = [c for c in FEATURES if c != "time (min)"]

df = pd.read_csv(PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

missing = set(FEATURES + [TARGET]) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

data = df[FEATURES + [TARGET]].copy()

if data.isna().any().any():
    raise ValueError("Missing values found.")

for column in CATEGORICAL:
    data[column] = data[column].astype(str).str.strip()

for column in NUMERIC + [TARGET]:
    data[column] = pd.to_numeric(data[column], errors="raise")

if not np.isfinite(data[NUMERIC + [TARGET]].to_numpy()).all():
    raise ValueError("Infinite or invalid numeric values found.")

if not data[TARGET].between(0, 100).all():
    raise ValueError("Efficiency must be between 0 and 100.")

if (data[NUMERIC] < 0).any().any():
    raise ValueError("Negative experimental values found.")

if not data["Ci (mg/L)"].gt(0).all():
    raise ValueError("Initial concentration must be positive.")

if not data["solution_pH"].between(0, 14).all():
    raise ValueError("Unexpected pH values found.")

X, y = data[FEATURES], data[TARGET]

groups = pd.util.hash_pandas_object(
    X[GROUP_COLUMNS], index=False
).to_numpy()

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=SEED
)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_groups, test_groups = groups[train_idx], groups[test_idx]

assert not set(train_groups).intersection(test_groups)

folds = min(5, len(np.unique(train_groups)))
if folds < 3 or len(X_test) < 2:
    raise ValueError("Insufficient training groups or testing observations.")

cv = GroupKFold(n_splits=folds)
splits = list(cv.split(X_train, y_train, train_groups))

preprocessor = ColumnTransformer(
    [
        ("numeric", "passthrough", NUMERIC),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL
        )
    ],
    sparse_threshold=0
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        importance_type="gain",
        random_state=SEED,
        n_jobs=1
    ))
])

parameters = {
    "model__n_estimators": [100, 200, 400, 600],
    "model__max_depth": [1, 2, 3, 4],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__reg_alpha": [0, 0.1, 1, 5],
    "model__reg_lambda": [1, 5, 10, 20],
    "model__gamma": [0, 0.1, 1]
}

search = RandomizedSearchCV(
    pipeline,
    parameters,
    n_iter=60,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    refit="MAE",
    cv=splits,
    random_state=SEED,
    n_jobs=-1,
    return_train_score=True,
    error_score="raise",
    verbose=1
)

search.fit(X_train, y_train)
model = search.best_estimator_

ptr = model.predict(X_train)
pte = model.predict(X_test)

oof = cross_val_predict(
    clone(model), X_train, y_train, cv=splits, n_jobs=-1
)

baseline = DummyRegressor(strategy="mean")
baseline.fit(np.zeros((len(y_train), 1)), y_train)
baseline_test = baseline.predict(np.zeros((len(y_test), 1)))

def evaluate(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = predicted - actual
    absolute = np.abs(error)
    nonzero = np.abs(actual) > 1e-12
    denominator = np.abs(actual) + np.abs(predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    span = np.ptp(actual)

    return {
        "R2": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mean_squared_error(actual, predicted),
        "RMSE": rmse,
        "Median_AE": median_absolute_error(actual, predicted),
        "Explained_Variance": explained_variance_score(actual, predicted),
        "MAPE_nonzero_percent": (
            100 * np.mean(absolute[nonzero] / np.abs(actual[nonzero]))
            if nonzero.any() else np.nan
        ),
        "SMAPE_percent": 100 * np.mean(np.divide(
            2 * absolute,
            denominator,
            out=np.zeros_like(absolute),
            where=denominator > 0
        )),
        "Mean_Bias": error.mean(),
        "Residual_SD": np.std(error, ddof=1),
        "Max_Absolute_Error": absolute.max(),
        "NRMSE_range_percent": 100 * rmse / span if span else np.nan,
        "Within_5_points_percent": 100 * np.mean(absolute <= 5),
        "Within_10_points_percent": 100 * np.mean(absolute <= 10),
        "Outside_0_100_percent": 100 * np.mean(
            (predicted < 0) | (predicted > 100)
        )
    }

scores = pd.DataFrame({
    "Training": evaluate(y_train, ptr),
    "Training_OOF": evaluate(y_train, oof),
    "Testing": evaluate(y_test, pte),
    "Dummy_Testing": evaluate(y_test, baseline_test)
}).T

scores.to_csv(OUT / "Tables" / "Evaluation_Metrics.csv")
pd.DataFrame(search.cv_results_).to_csv(
    OUT / "Tables" / "Hyperparameter_Search.csv", index=False
)

split_table = pd.DataFrame({
    "Original_CSV_Line": np.arange(len(df)) + 2,
    "Group": groups.astype(str),
    "Split": "Training"
})
split_table.loc[test_idx, "Split"] = "Testing"
split_table.to_csv(OUT / "Tables" / "Data_Split.csv", index=False)

df.loc[df.duplicated(keep=False)].to_csv(
    OUT / "Tables" / "Duplicate_Records_For_Review.csv", index=True
)

for label, features, actual, predicted in [
    ("Training", X_train, y_train, ptr),
    ("Training_OOF", X_train, y_train, oof),
    ("Testing", X_test, y_test, pte)
]:
    table = features.copy()
    table.insert(0, "Original_CSV_Line", features.index + 2)
    table["Actual"] = actual.to_numpy()
    table["Predicted"] = predicted
    table["Residual"] = actual.to_numpy() - predicted
    table.to_csv(
        OUT / "Tables" / f"{label}_Predictions.csv", index=False
    )

joblib.dump(
    {"pipeline": model, "features": FEATURES, "target": TARGET},
    OUT / "Models" / "XGBoost_Pipeline.joblib"
)
model.named_steps["model"].save_model(
    str(OUT / "Models" / "XGBoost.json")
)

metadata = {
    "features": FEATURES,
    "target": TARGET,
    "excluded_outcome": "Cf (mg/L)",
    "grouping_columns": GROUP_COLUMNS,
    "training_rows": len(X_train),
    "testing_rows": len(X_test),
    "training_groups": len(np.unique(train_groups)),
    "testing_groups": len(np.unique(test_groups)),
    "exact_duplicate_rows_retained": int(df.duplicated().sum()),
    "best_parameters": search.best_params_,
    "seed": SEED
}
with open(OUT / "Models" / "Metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

fold_records, permutation_records = [], []

for number, (a, b) in enumerate(splits, 1):
    fold_model = clone(model).fit(X_train.iloc[a], y_train.iloc[a])
    prediction = fold_model.predict(X_train.iloc[b])

    fold_records.append({
        "Fold": number,
        **evaluate(y_train.iloc[b], prediction)
    })

    result = permutation_importance(
        fold_model, X_train.iloc[b], y_train.iloc[b],
        scoring="neg_mean_absolute_error",
        n_repeats=15, random_state=SEED, n_jobs=-1
    )
    permutation_records.append(result.importances_mean)

fold_table = pd.DataFrame(fold_records)
fold_table.to_csv(OUT / "Tables" / "CV_Fold_Metrics.csv", index=False)

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Mean_MAE_Increase": np.mean(permutation_records, axis=0),
    "Fold_SD": np.std(permutation_records, axis=0, ddof=1)
}).sort_values("Mean_MAE_Increase")

importance.to_csv(
    OUT / "Tables" / "CV_Permutation_Importance.csv", index=False
)

learning_records = []

for fraction in np.linspace(0.3, 1.0, 5):
    train_errors, valid_errors, row_counts = [], [], []

    for a, b in splits:
        available = np.unique(train_groups[a])
        ordered = np.random.default_rng(SEED).permutation(available)
        count = max(1, int(np.ceil(len(ordered) * fraction)))
        selected = a[np.isin(train_groups[a], ordered[:count])]

        fold_model = clone(model).fit(
            X_train.iloc[selected], y_train.iloc[selected]
        )
        train_errors.append(mean_absolute_error(
            y_train.iloc[selected],
            fold_model.predict(X_train.iloc[selected])
        ))
        valid_errors.append(mean_absolute_error(
            y_train.iloc[b],
            fold_model.predict(X_train.iloc[b])
        ))
        row_counts.append(len(selected))

    learning_records.append({
        "Fraction": fraction,
        "Mean_Training_Rows": np.mean(row_counts),
        "Training_MAE": np.mean(train_errors),
        "Validation_MAE": np.mean(valid_errors),
        "Validation_SD": np.std(valid_errors, ddof=1)
    })

learning = pd.DataFrame(learning_records)
learning.to_csv(OUT / "Tables" / "Learning_Curve.csv", index=False)

prep = model.named_steps["preprocess"]
encoded = np.asarray(prep.transform(X_train), dtype=float)
encoded_names = list(NUMERIC) + list(
    prep.named_transformers_["categorical"].get_feature_names_out(
        CATEGORICAL
    )
)
encoded_df = pd.DataFrame(
    encoded, columns=encoded_names, index=X_train.index
)

explainer = shap.TreeExplainer(model.named_steps["model"])
explanation = explainer(encoded_df)
shap_values = explanation.values

pd.DataFrame(
    shap_values, columns=encoded_names, index=X_train.index
).to_csv(OUT / "Tables" / "Training_SHAP_Values.csv")

plt.rcParams.update({
    "figure.figsize": (9, 6),
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.2
})

saved = []

def save(number, title):
    fig = plt.gcf()
    fig.tight_layout()
    stem = f"{number:02d}_{title.replace(' ', '_')}"
    for extension in ["png", "pdf"]:
        fig.savefig(
            OUT / "Figures" / f"{stem}.{extension}",
            dpi=300, bbox_inches="tight"
        )
    saved.append(stem)
    plt.close(fig)

def parity(ax, observed, predicted, title):
    low = min(np.min(observed), np.min(predicted))
    high = max(np.max(observed), np.max(predicted))
    ax.scatter(observed, predicted, alpha=0.6)
    ax.plot([low, high], [low, high], "k--")
    ax.set(
        xlabel="Actual efficiency (%)",
        ylabel="Predicted efficiency (%)",
        title=title
    )

actual = y_test.to_numpy()
residual = actual - pte
absolute = np.abs(residual)

fig, ax = plt.subplots()
parity(ax, y_train, ptr, "XGBoost Training")
save(1, "Training Parity")

fig, ax = plt.subplots()
parity(ax, y_train, oof, "Training Out-of-Fold Predictions")
save(2, "Training OOF Parity")

fig, ax = plt.subplots()
parity(
    ax, actual, pte,
    f"Testing R² = {scores.loc['Testing', 'R2']:.3f}"
)
save(3, "Testing Parity")

plt.figure()
plt.scatter(y_train, ptr, alpha=0.3, label="Training")
plt.scatter(actual, pte, alpha=0.7, label="Testing")
limits = [
    min(y.min(), ptr.min(), pte.min()),
    max(y.max(), ptr.max(), pte.max())
]
plt.plot(limits, limits, "k--")
plt.xlabel("Actual efficiency (%)")
plt.ylabel("Predicted efficiency (%)")
plt.legend()
save(4, "Training Testing Comparison")

plt.figure()
plt.scatter(pte, residual)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Predicted efficiency (%)")
plt.ylabel("Residual")
save(5, "Residual Versus Prediction")

plt.figure()
plt.scatter(actual, residual)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Actual efficiency (%)")
plt.ylabel("Residual")
save(6, "Residual Versus Actual")

plt.figure()
plt.hist(residual, bins="auto", edgecolor="white")
plt.xlabel("Residual")
plt.ylabel("Count")
save(7, "Residual Histogram")

plt.figure()
stats.probplot(residual, dist="norm", plot=plt)
save(8, "Residual QQ Plot")

order = np.argsort(actual)
plt.figure(figsize=(11, 6))
plt.plot(actual[order], label="Actual")
plt.plot(pte[order], label="Predicted")
plt.xlabel("Test observations sorted by actual efficiency")
plt.ylabel("Efficiency (%)")
plt.legend()
save(9, "Sorted Testing Predictions")

plt.figure()
plt.scatter(actual, absolute)
plt.xlabel("Actual efficiency (%)")
plt.ylabel("Absolute error")
save(10, "Absolute Error Versus Actual")

plt.figure()
plt.step(
    np.sort(absolute),
    np.arange(1, len(absolute) + 1) / len(absolute),
    where="post"
)
plt.xlabel("Absolute error")
plt.ylabel("Cumulative fraction")
save(11, "Absolute Error ECDF")

plt.figure()
bins = np.linspace(
    min(actual.min(), pte.min()),
    max(actual.max(), pte.max()), 16
)
plt.hist(actual, bins=bins, alpha=0.6, label="Actual")
plt.hist(pte, bins=bins, alpha=0.6, label="Predicted")
plt.xlabel("Efficiency (%)")
plt.ylabel("Count")
plt.legend()
save(12, "Testing Target Distributions")

plt.figure()
plt.boxplot([
    np.abs(y_train.to_numpy() - ptr),
    np.abs(y_train.to_numpy() - oof),
    absolute,
    np.abs(actual - baseline_test)
])
plt.xticks(
    [1, 2, 3, 4],
    ["Training", "Training OOF", "Testing", "Dummy Testing"]
)
plt.ylabel("Absolute error")
save(13, "Error Distribution Comparison")

plt.figure()
plt.scatter(pte, np.sqrt(absolute))
plt.xlabel("Predicted efficiency (%)")
plt.ylabel("Square root of absolute residual")
save(14, "Residual Scale Plot")

fig, axes = plt.subplots(3, 3, figsize=(17, 13))
for ax, column in zip(axes.flat, NUMERIC):
    ax.scatter(X_test[column], residual, alpha=0.6)
    ax.axhline(0, color="black", linestyle="--")
    ax.set(xlabel=column, ylabel="Residual")
axes.flat[-1].axis("off")
save(15, "Numeric Feature Residuals")

correlation = X_train[NUMERIC].assign(efficiency=y_train).corr()
correlation.to_csv(OUT / "Tables" / "Training_Correlation.csv")

plt.figure(figsize=(12, 10))
plt.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Pearson correlation")
plt.xticks(
    range(len(correlation)), correlation.columns,
    rotation=65, ha="right"
)
plt.yticks(range(len(correlation)), correlation.columns)
for i in range(len(correlation)):
    for j in range(len(correlation)):
        plt.text(
            j, i, f"{correlation.iloc[i, j]:.2f}",
            ha="center", va="center", fontsize=8
        )
save(16, "Training Correlation Matrix")

fig, axes = plt.subplots(3, 3, figsize=(17, 13))
for ax, column in zip(axes.flat, NUMERIC):
    bins = np.histogram_bin_edges(X[column], bins=12)
    ax.hist(
        X_train[column], bins=bins, density=True,
        alpha=0.6, label="Training"
    )
    ax.hist(
        X_test[column], bins=bins, density=True,
        alpha=0.6, label="Testing"
    )
    ax.set(xlabel=column, ylabel="Density")
    ax.legend()
axes.flat[-1].axis("off")
save(17, "Numeric Feature Distributions")

plt.figure(figsize=(10, 7))
plt.barh(
    importance.Feature, importance.Mean_MAE_Increase,
    xerr=importance.Fold_SD, capsize=3
)
plt.xlabel("Validation MAE increase after permutation")
save(18, "Cross Validated Permutation Importance")

gain = pd.Series(
    model.named_steps["model"].feature_importances_,
    index=encoded_names
).sort_values()

gain.to_csv(OUT / "Tables" / "Encoded_Gain_Importance.csv")
plt.figure(figsize=(10, 10))
gain.tail(20).plot.barh()
plt.xlabel("Normalized gain importance")
save(19, "XGBoost Gain Importance")

plt.figure()
plt.plot(
    learning.Mean_Training_Rows,
    learning.Training_MAE, "o-", label="Training"
)
plt.plot(
    learning.Mean_Training_Rows,
    learning.Validation_MAE, "o-", label="Validation"
)
plt.fill_between(
    learning.Mean_Training_Rows.to_numpy(),
    (learning.Validation_MAE - learning.Validation_SD).to_numpy(),
    (learning.Validation_MAE + learning.Validation_SD).to_numpy(),
    alpha=0.2
)
plt.xlabel("Mean training observations")
plt.ylabel("MAE")
plt.legend()
save(20, "Grouped Learning Curve")

plt.figure()
plt.bar(fold_table.Fold, fold_table.MAE)
plt.xlabel("Training validation fold")
plt.ylabel("MAE")
save(21, "Validation Fold Stability")

shap.summary_plot(
    shap_values, encoded_df, max_display=20, show=False
)
save(22, "Training SHAP Beeswarm")

shap.summary_plot(
    shap_values, encoded_df,
    plot_type="bar", max_display=20, show=False
)
save(23, "Training SHAP Importance")

fig, axes = plt.subplots(2, 4, figsize=(21, 10))
PartialDependenceDisplay.from_estimator(
    model, X_train, features=NUMERIC,
    method="brute", ax=list(axes.flat)
)
save(24, "Training Numeric Partial Dependence")

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
parity(axes[0, 0], y_train, ptr, "Training Fit")
parity(axes[0, 1], y_train, oof, "Training Out-of-Fold Predictions")
parity(axes[1, 0], actual, pte, "Held-Out Testing")
axes[1, 1].axis("off")
axes[1, 1].text(
    0.03, 0.95,
    f"Model: XGBoost\n\n"
    f"Training rows: {len(X_train)}\n"
    f"Testing rows: {len(X_test)}\n"
    f"Training groups: {len(np.unique(train_groups))}\n"
    f"Testing groups: {len(np.unique(test_groups))}\n\n"
    f"Best CV MAE: {-search.best_score_:.3f}\n"
    f"Training R²: {scores.loc['Training', 'R2']:.3f}\n"
    f"OOF R²: {scores.loc['Training_OOF', 'R2']:.3f}\n"
    f"Test R²: {scores.loc['Testing', 'R2']:.3f}\n"
    f"Test MAE: {scores.loc['Testing', 'MAE']:.3f}\n"
    f"Test RMSE: {scores.loc['Testing', 'RMSE']:.3f}\n\n"
    f"Dummy test MAE: {scores.loc['Dummy_Testing', 'MAE']:.3f}",
    va="top", fontsize=12
)
save(25, "Training Validation Testing Dashboard")

assert len(saved) == 25

pd.DataFrame({
    "Number": range(1, 26),
    "Figure": saved
}).to_csv(OUT / "Tables" / "Figure_Index.csv", index=False)

print("\nBest parameters:")
print(json.dumps(search.best_params_, indent=2))
print("\nEvaluation metrics:")
print(scores.round(4).to_string())
print(f"\nDuplicate rows retained: {df.duplicated().sum()}")
print(f"Saved {len(saved)} graphs in PNG and PDF formats.")
print(f"Results directory: {OUT}")

Fitting 5 folds for each of 60 candidates, totalling 300 fits

Best parameters:
{
  "model__subsample": 0.85,
  "model__reg_lambda": 1,
  "model__reg_alpha": 0.1,
  "model__n_estimators": 200,
  "model__min_child_weight": 5,
  "model__max_depth": 3,
  "model__learning_rate": 0.1,
  "model__gamma": 1,
  "model__colsample_bytree": 0.7
}

Evaluation metrics:
                   R2      MAE        MSE     RMSE  Median_AE  Explained_Variance  MAPE_nonzero_percent  SMAPE_percent  Mean_Bias  Residual_SD  Max_Absolute_Error  NRMSE_range_percent  Within_5_points_percent  Within_10_points_percent  Outside_0_100_percent
Training       0.9861   2.7792    14.2785   3.7787     2.1690              0.9861               21.3812        30.3623    -0.0021       3.7807             19.6880               3.7787                  86.0417                   97.9167                 8.4375
Training_OOF   0.6447  13.3921   364.3434  19.0878     7.7579              0.6486              149.9402        52.8440     1.9

In [4]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

VALIDATION_OUT = OUT / "Nested_Validation"
VALIDATION_OUT.mkdir(parents=True, exist_ok=True)

base_X = X_train.reset_index(drop=True)
base_y = y_train.reset_index(drop=True)
base_groups = np.asarray(train_groups)
source_lines = X_train.index.to_numpy() + 2

unique_groups = np.unique(base_groups)
shuffled_groups = np.random.default_rng(SEED).permutation(unique_groups)
outer_group_sets = np.array_split(shuffled_groups, 5)

original_training = df.loc[X_train.index]
keep_unique = ~original_training.duplicated().to_numpy()

scenarios = {
    "Duplicates_Retained": np.ones(len(base_X), dtype=bool),
    "Duplicates_Removed": keep_unique
}

fold_results = []
prediction_results = []
parameter_results = []

for scenario, keep in scenarios.items():
    for fold, held_groups in enumerate(outer_group_sets, 1):
        outer_train = np.flatnonzero(
            keep & ~np.isin(base_groups, held_groups)
        )
        outer_valid = np.flatnonzero(
            keep & np.isin(base_groups, held_groups)
        )

        xa = base_X.iloc[outer_train]
        ya = base_y.iloc[outer_train]
        xb = base_X.iloc[outer_valid]
        yb = base_y.iloc[outer_valid]
        ga = base_groups[outer_train]

        inner = GroupKFold(
            n_splits=min(4, len(np.unique(ga)))
        )

        xgb_search = RandomizedSearchCV(
            clone(pipeline),
            parameters,
            n_iter=40,
            scoring="neg_mean_absolute_error",
            cv=inner,
            random_state=SEED,
            n_jobs=-1,
            error_score="raise"
        )
        xgb_search.fit(xa, ya, groups=ga)

        ridge = Pipeline([
            ("preprocess", clone(preprocessor)),
            ("scale", StandardScaler()),
            ("model", Ridge())
        ])

        ridge_search = GridSearchCV(
            ridge,
            {"model__alpha": [0.1, 1, 10, 100, 1000]},
            scoring="neg_mean_absolute_error",
            cv=inner,
            n_jobs=-1,
            error_score="raise"
        )
        ridge_search.fit(xa, ya, groups=ga)

        candidates = {
            "XGBoost": xgb_search.best_estimator_,
            "Ridge": ridge_search.best_estimator_,
            "Mean": DummyRegressor(strategy="mean").fit(xa, ya),
            "Median": DummyRegressor(strategy="median").fit(xa, ya)
        }

        parameter_results.append({
            "Scenario": scenario,
            "Fold": fold,
            "XGBoost": xgb_search.best_params_,
            "Ridge": ridge_search.best_params_
        })

        for name, fitted_model in candidates.items():
            predicted = fitted_model.predict(xb)

            fold_results.append({
                "Scenario": scenario,
                "Model": name,
                "Fold": fold,
                "Training_Rows": len(xa),
                "Validation_Rows": len(xb),
                "Validation_Groups": len(np.unique(base_groups[outer_valid])),
                **evaluate(yb, predicted)
            })

            rows = xb.copy()
            rows["Scenario"] = scenario
            rows["Model"] = name
            rows["Fold"] = fold
            rows["Original_CSV_Line"] = source_lines[outer_valid]
            rows["Group"] = base_groups[outer_valid].astype(str)
            rows["Actual"] = yb.to_numpy()
            rows["Predicted"] = predicted
            rows["Residual"] = rows["Actual"] - predicted
            rows["Absolute_Error"] = np.abs(rows["Residual"])
            prediction_results.append(rows)

        print(f"{scenario}: outer fold {fold}/5 completed")

fold_metrics = pd.DataFrame(fold_results)
predictions = pd.concat(prediction_results, ignore_index=True)

fold_metrics.to_csv(
    VALIDATION_OUT / "Nested_Fold_Metrics.csv", index=False
)
predictions.to_csv(
    VALIDATION_OUT / "Nested_Predictions.csv", index=False
)

with open(VALIDATION_OUT / "Nested_Best_Parameters.json", "w") as f:
    json.dump(parameter_results, f, indent=2)

summary_rows = []

for (scenario, name), rows in predictions.groupby(["Scenario", "Model"]):
    folds_for_model = fold_metrics[
        (fold_metrics["Scenario"] == scenario)
        & (fold_metrics["Model"] == name)
    ]

    group_mae = rows.groupby("Group")["Absolute_Error"].mean()

    summary_rows.append({
        "Scenario": scenario,
        "Model": name,
        "Rows": len(rows),
        "Groups": rows["Group"].nunique(),
        "Pooled_R2": r2_score(rows["Actual"], rows["Predicted"]),
        "Pooled_MAE": mean_absolute_error(
            rows["Actual"], rows["Predicted"]
        ),
        "Pooled_RMSE": np.sqrt(mean_squared_error(
            rows["Actual"], rows["Predicted"]
        )),
        "Mean_Fold_MAE": folds_for_model["MAE"].mean(),
        "SD_Fold_MAE": folds_for_model["MAE"].std(),
        "Mean_Group_MAE": group_mae.mean(),
        "Mean_Bias": (rows["Predicted"] - rows["Actual"]).mean()
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(VALIDATION_OUT / "Nested_Summary.csv", index=False)

xgb_predictions = predictions[predictions["Model"] == "XGBoost"].copy()

def breakdown(frame, keys):
    records = []

    for values, rows in frame.groupby(keys, dropna=False):
        values = values if isinstance(values, tuple) else (values,)
        record = dict(zip(keys, values))
        record.update({
            "Rows": len(rows),
            "Groups": rows["Group"].nunique(),
            "MAE": rows["Absolute_Error"].mean(),
            "RMSE": np.sqrt(np.mean(rows["Residual"] ** 2)),
            "Bias": (rows["Predicted"] - rows["Actual"]).mean(),
            "Max_Error": rows["Absolute_Error"].max(),
            "Within_10_points_percent": 100 * np.mean(
                rows["Absolute_Error"] <= 10
            )
        })
        records.append(record)

    return pd.DataFrame(records).sort_values(
        ["Scenario", "MAE"], ascending=[True, False]
    )

catalyst_metrics = breakdown(
    xgb_predictions, ["Scenario", "Catalyst_type"]
)
group_metrics = breakdown(
    xgb_predictions, ["Scenario", "Group"]
)

group_conditions = (
    xgb_predictions[["Scenario", "Group"] + GROUP_COLUMNS]
    .drop_duplicates(["Scenario", "Group"])
)
group_metrics = group_metrics.merge(
    group_conditions, on=["Scenario", "Group"], how="left"
)

catalyst_metrics.to_csv(
    VALIDATION_OUT / "Errors_By_Catalyst.csv", index=False
)
group_metrics.to_csv(
    VALIDATION_OUT / "Errors_By_Condition_Group.csv", index=False
)

xgb_predictions.sort_values(
    "Absolute_Error", ascending=False
).head(50).to_csv(
    VALIDATION_OUT / "Largest_50_Errors.csv", index=False
)

matched_summary = []

for scenario in scenarios:
    rows = predictions[predictions["Scenario"] == scenario]
    rows = rows[
        rows["Original_CSV_Line"].isin(source_lines[keep_unique])
    ]

    for name, subset in rows.groupby("Model"):
        matched_summary.append({
            "Scenario": scenario,
            "Model": name,
            "Common_Evaluation_Rows": len(subset),
            "MAE": mean_absolute_error(
                subset["Actual"], subset["Predicted"]
            ),
            "RMSE": np.sqrt(mean_squared_error(
                subset["Actual"], subset["Predicted"]
            )),
            "R2": r2_score(subset["Actual"], subset["Predicted"])
        })

matched = pd.DataFrame(matched_summary)
matched.to_csv(
    VALIDATION_OUT / "Duplicate_Sensitivity_Common_Rows.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(10, 6))
summary.pivot(
    index="Model", columns="Scenario", values="Pooled_MAE"
).plot.bar(ax=ax)
ax.set_ylabel("Nested validation MAE")
fig.tight_layout()
fig.savefig(
    VALIDATION_OUT / "Nested_Model_Comparison.png", dpi=300
)
plt.close(fig)

print("\nNested grouped validation:")
print(summary.round(4).to_string(index=False))
print("\nDuplicate sensitivity on identical evaluation rows:")
print(matched.round(4).to_string(index=False))
print("\nHighest-error catalyst groups:")
print(catalyst_metrics.head(10).round(4).to_string(index=False))
print(f"\nSaved to: {VALIDATION_OUT}")

Duplicates_Retained: outer fold 1/5 completed
Duplicates_Retained: outer fold 2/5 completed
Duplicates_Retained: outer fold 3/5 completed
Duplicates_Retained: outer fold 4/5 completed
Duplicates_Retained: outer fold 5/5 completed
Duplicates_Removed: outer fold 1/5 completed
Duplicates_Removed: outer fold 2/5 completed
Duplicates_Removed: outer fold 3/5 completed
Duplicates_Removed: outer fold 4/5 completed
Duplicates_Removed: outer fold 5/5 completed

Nested grouped validation:
           Scenario   Model  Rows  Groups  Pooled_R2  Pooled_MAE  Pooled_RMSE  Mean_Fold_MAE  SD_Fold_MAE  Mean_Group_MAE  Mean_Bias
 Duplicates_Removed    Mean   826      31    -0.0157     25.2237      29.3644        25.2442       1.8703         25.6006    -0.0656
 Duplicates_Removed  Median   826      31    -0.0192     25.2622      29.4146        25.2789       1.7060         25.6456    -0.8529
 Duplicates_Removed   Ridge   826      31     0.7115     11.7917      15.6501        11.7773       2.3269         11.9